<a href="https://colab.research.google.com/github/pritinandabarpanda2006-cloud/2502110069_IT12/blob/main/LAB09_2502110069_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IT-12

In [ ]:
import os

os.makedirs("data", exist_ok=True)

data = [f"Record {i}" for i in range(12)] + [
    "WO-08 ERROR: \x00\xff BAD_HEADER",
    "WO-08 ERROR: NULL_TIMESTAMP",
    "WO-08 ERROR: CORRUPT_PAYLOAD $$$",
]

path = "data/shift_log.txt"
with open(path, "w") as f:
    f.write("\n".join(data) + "\n")

print(f"Line count: {len(open(path).readlines())}")

Line count: 15


IT-10

In [ ]:
import csv, re

v, r = [], []
for line in open("data/shift_log.txt"):
    raw = line.strip()
    p = [x.strip() for x in raw.split("|")]
    if len(p) != 3: r.append([raw, "Bad format"])
    elif not re.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$", p[0]): r.append([raw, "Bad time"])
    elif any(k in raw for k in ["CORRUPT", "ERROR", "$"]): r.append([raw, "Corrupt"])
    else: v.append(p)

open("data/report.txt", "w").write("TIMESTAMP          | WORK ORDER | DETAILS\n" + "-"*45 + "\n" + "".join(f"{ts:<18} | {wo:<10} | {dt}\n" for ts, wo, dt in v))
csv.writer(open("data/rejects.csv", "w", newline="")).writerows([["line", "reason"]] + r)

IT-8

In [ ]:
import csv, re

# Regex handled irregular spacing better than split()
P = re.compile(r"^\s*(\d{4}-\d{2}-\d{2} \d{2}:\d{2})\s*\|\s*(WO-\d+)\s*\|\s*([^|$]+)\s*$")
v, r = [], []

for line in open("data/shift_log.txt"):
    m = P.match(line.strip())
    (v if m else r).append(m.groups() if m else [line.strip(), "Bad regex match"])

open("data/report.txt", "w").write("TIMESTAMP          | WORK ORDER | DETAILS\n" + "-"*45 + "\n" + "".join(f"{ts:<18} | {wo:<10} | {dt}\n" for ts, wo, dt in v))
csv.writer(open("data/rejects.csv", "w", newline="")).writerows([["line", "reason"]] + r)

# OLD: p = line.split("|"); v.append(p) if len(p)==3 else r.append(...)

IT-6

In [ ]:
import csv, glob, re, datetime as D

P = re.compile(
    r"^\s*(\d{4}-\d{2}-\d{2} \d{2}:\d{2})\s*\|\s*(WO-\d+)\s*\|\s*([^|$]+)\s*$"
)
A, H = [], []

for f in glob.glob("data/*.txt"):
    L = [l.strip() for l in open(f)]
    V = [P.match(l).groups() for l in L if P.match(l)]
    R = [[l, "Err"] for l in L if not P.match(l)]
    open(f + ".rep", "w").write(
        "".join(f"{t:<18}| {w:<10}| {d}\n" for t, w, d in V)
    )
    csv.writer(open(f + ".rej", "w", newline="")).writerows(R)
    A += V
    H.append([D.datetime.now(), f, len(L), len(R)])

open("data/summary.txt", "w").write(
    f"Total:{len(A)}\n" + "".join(f"{t:<18}| {w:<10}| {d}\n" for t, w, d in A)
)
csv.writer(open("data/run_history.csv", "a", newline="")).writerows(H)